# 33 — Efficient Training, Inference, and Model Deployment

A useful model must be correct, reproducible, and practical to run.

We will study:

- DataLoader efficiency
- Pinned memory
- Non-blocking transfers
- Mixed precision review
- `torch.compile`
- Profiling
- Inference batching
- Latency and throughput
- Model size
- Export
- Deployment validation
- Reproducible inference pipelines


In [ ]:
import time
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader
print("PyTorch:",torch.__version__)


# 1. Training vs Inference Efficiency

Training cares about:

- Samples/sec
- GPU utilization
- Memory

Inference cares about:

- Latency
- Throughput
- Memory
- Model size


In [ ]:
class EfficientCNN(nn.Module):
    def __init__(self,num_classes=3):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),nn.Linear(32,num_classes)
        )
    def forward(self,x): return self.net(x)


# 2. DataLoader Efficiency


In [ ]:
images=torch.randn(1000,1,64,64)
targets=torch.randint(0,3,(1000,))
dataset=TensorDataset(images,targets)

loader=DataLoader(dataset,batch_size=64,shuffle=True,num_workers=0)
print(len(loader))


# 3. Pinned Memory

With CUDA, `pin_memory=True` can speed CPU→GPU transfer.


In [ ]:
use_cuda=torch.cuda.is_available()
loader=DataLoader(
    dataset,batch_size=64,shuffle=True,
    num_workers=0,pin_memory=use_cuda
)


# 4. Non-Blocking Transfers


In [ ]:
device=torch.device("cuda" if use_cuda else "cpu")
x,y=next(iter(loader))
x=x.to(device,non_blocking=use_cuda)
y=y.to(device,non_blocking=use_cuda)


# 5. Mixed Precision Review

On CUDA, autocast can reduce memory and increase speed.


In [ ]:
model=EfficientCNN().to(device)
if device.type=="cuda":
    with torch.autocast(device_type="cuda",dtype=torch.float16):
        out=model(x)
else:
    out=model(x)
print(out.shape)


# 6. `torch.compile`

Compilation may improve performance after warm-up, depending on model and hardware.


In [ ]:
def maybe_compile(model):
    if hasattr(torch,"compile"):
        try:
            return torch.compile(model)
        except Exception as e:
            print("Compile unavailable:",e)
    return model


# 7. Warm-Up

Benchmark after warm-up because first runs may include initialization/compilation overhead.


In [ ]:
def warmup(model,x,n=5):
    model.eval()
    with torch.inference_mode():
        for _ in range(n): _=model(x)


# 8. Latency Benchmarking


In [ ]:
def benchmark_latency(model,x,iterations=30):
    model.eval()
    if x.device.type=="cuda": torch.cuda.synchronize()
    start=time.perf_counter()
    with torch.inference_mode():
        for _ in range(iterations): _=model(x)
    if x.device.type=="cuda": torch.cuda.synchronize()
    return (time.perf_counter()-start)/iterations


# 9. Throughput

$$
Throughput=\frac{BatchSize}{Latency}
$$


In [ ]:
def throughput(batch_size,latency):
    return batch_size/latency


# 10. Batch Size Tradeoff

Larger batches often increase throughput but may increase latency and memory.


# 11. `torch.inference_mode()`

Use for inference:

```python
model.eval()
with torch.inference_mode():
    ...
```


# 12. Parameter Count and Weight Size


In [ ]:
def parameter_count(model):
    return sum(p.numel() for p in model.parameters())

params=parameter_count(model)
print("Parameters:",params)
print("Approx FP32 MB:",params*4/(1024**2))


# 13. Training Memory

Memory includes:

- Parameters
- Activations
- Gradients
- Optimizer state


# 14. Gradient Checkpointing

Trades compute for memory by recomputing activations during backward.


# 15. Profiling


In [ ]:
def profile_once(model,x):
    activities=[torch.profiler.ProfilerActivity.CPU]
    if x.device.type=="cuda":
        activities.append(torch.profiler.ProfilerActivity.CUDA)
    with torch.profiler.profile(activities=activities,record_shapes=True) as prof:
        with torch.inference_mode(): _=model(x)
    return prof


# 16. Optimize the Bottleneck

Profile first. Do not spend time optimizing code that is not limiting performance.


# 17. `torch.export` Concept

Modern PyTorch provides export APIs for capturing models for downstream runtimes.


In [ ]:
def try_export(model,example):
    if hasattr(torch,"export"):
        try:
            return torch.export.export(model,(example,))
        except Exception as e:
            print("Export failed:",e)
    return None


# 18. ONNX

ONNX can support deployment outside PyTorch. Always verify operator support and numerical equivalence.


# 19. Numerical Equivalence


In [ ]:
def max_abs_diff(a,b):
    return float((a-b).abs().max())


# 20. Reproducible Inference Pipeline

Deployment needs:

- Weights
- Input size
- Channel convention
- Normalization
- Class mapping
- Threshold
- Patient aggregation rule


# 21. Preprocessing Must Match Training

A deployment pipeline with different normalization is a different model system.


# 22. CPU vs GPU

GPU favors throughput. CPU may simplify deployment and be sufficient for small models.


# 23. Edge Deployment

Portable ultrasound systems may require small models, low memory, and low latency.


# 24. Quantization Intuition

Lower-precision inference can reduce size and improve speed, but must be revalidated for accuracy/calibration.


# 25. Dynamic Shapes

Variable image dimensions complicate export and serving. Standardized inputs often simplify deployment.


# 26. Input Validation


In [ ]:
def validate_input(x):
    assert x.ndim==4
    assert x.shape[1]==1
    assert x.dtype in {torch.float16,torch.float32,torch.float64}


# 27. Post-Deployment Monitoring

Monitor:

- Input distribution
- Site/device
- Missingness
- Confidence
- Failure rate
- Drift


# 28. Common Mistakes

- Benchmarking the first run
- Forgetting CUDA synchronization
- Measuring model-only latency when preprocessing dominates
- Deploying with different preprocessing
- Assuming quantization/export preserves all behavior


# 29. Exercises

1. Benchmark DataLoader settings.
2. Measure latency.
3. Compute throughput.
4. Compare batch sizes.
5. Try `torch.compile`.
6. Profile one inference.
7. Estimate model size.
8. Try export.
9. Verify numerical equivalence.
10. Build an inference config.


# 30. Key Takeaways

Optimization order:

$$
\boxed{Correctness\rightarrow Profiling\rightarrow Optimization}
$$

Deployment requires:

$$
\boxed{Model+Preprocessing+Decision\ Rules}
$$


# Next Notebook

# 34 — Reproducible Research Pipelines, Experiment Tracking, and Packaging

In the next notebook, we will move from exploratory notebooks to a reproducible research project structure.
